# Seq2Seq: 전처리 모듈 → 분석 그래프 → 학습 → 대화
`esg` 커널에서 이 노트북만 실행하세요. `preprocessing.py`를 모듈로 불러 전처리합니다.
MODE="train"은 전처리 및 학습, MODE="chat"은 저장 모델로 대화합니다.
PREPARED_DIR에 기존 결과 폴더를 지정하면 전처리를 재사용합니다.

In [ ]:
import os
import sys
from pathlib import Path
if Path(sys.prefix).name.lower() != "esg":
    raise RuntimeError(f"esg 커널을 선택하세요. 현재 Python: {sys.executable}")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import json
import numpy as np
import tensorflow as tf
print("Python:", sys.executable, "TensorFlow:", tf.__version__)
tf.keras.utils.set_random_seed(1234)

MODE = "train"  # 저장 모델 테스트만 하려면 "chat"
EPOCHS = 20
BATCH_SIZE = 64
UNITS = 128
EMBEDDING_DIM = 64

BASE = Path.cwd()
if (BASE / "ESG Chat Bot").is_dir():
    BASE = BASE / "ESG Chat Bot"
if not (BASE / "seq2seq.ipynb").exists():
    raise RuntimeError("프로젝트 루트 또는 ESG Chat Bot 폴더에서 실행하세요.")
sys.path.insert(0, str(BASE))
from preprocessing import PAD, SOS, EOS, UNK, encode, prepare
DATA_PATHS = [BASE.parent / "ChatbotData.csv", BASE.parent / "ESG_QnA_dataset_10000.csv"]
MAX_LENGTH = 256
MAX_SAMPLES = None  # 빠른 테스트: 128
PREPARED_DIR = None  # None: 새 전처리 실행. 재사용하려면 기존 폴더 지정
PREPROCESS_ROOT = BASE / "data_in"
RUN_DIR = BASE / "data_out" / "seq2seq_char"
WEIGHTS = RUN_DIR / "best.weights.h5"
CONFIG = RUN_DIR / "config.json"
assert MODE in {"train", "chat"}

## 1. 전처리 모듈 호출 및 분석 결과
데이터 통합·정제·어휘 생성은 preprocessing.py가 담당하며 그래프와 통계는 이 노트북에 표시합니다.

In [ ]:
if MODE == "train":
    from datetime import datetime
    if PREPARED_DIR is None:
        PREPARED_DIR, _ = prepare(DATA_PATHS, PREPROCESS_ROOT, MAX_LENGTH, MAX_SAMPLES)
    PREPARED_DIR = Path(PREPARED_DIR)
    config = json.loads((PREPARED_DIR / "preprocess_config.json").read_text(encoding="utf-8"))
    with np.load(PREPARED_DIR / "training_data.npz", allow_pickle=False) as arrays:
        x, y, decoder_inputs = arrays["x"], arrays["y"], arrays["decoder_inputs"]
    split = config["split"]
    if not (x.shape == y.shape == decoder_inputs.shape == (config["sample_count"], config["max_length"]) and 0 < split < len(x)):
        raise ValueError("전처리 배열과 설정이 일치하지 않습니다. 전처리를 다시 실행하세요.")
    config.update(units=UNITS, embedding_dim=EMBEDDING_DIM, prepared_dir=str(PREPARED_DIR))
    vocabulary = config["vocabulary"]
    RUN_DIR = RUN_DIR / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    WEIGHTS, CONFIG = RUN_DIR / "best.weights.h5", RUN_DIR / "config.json"
    CONFIG.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
    print("전처리 결과:", PREPARED_DIR)
    print(f"학습 {split:,} / 검증 {len(x)-split:,}")
else:
    # 특정 학습 결과를 선택하려면 아래 자동 선택 대신 RUN_DIR을 직접 지정하세요.
    saved = sorted(p.parent for p in RUN_DIR.glob("*/best.weights.h5") if (p.parent / "config.json").exists())
    if not saved:
        raise FileNotFoundError("저장 모델이 없습니다. 먼저 MODE='train'으로 학습하세요.")
    RUN_DIR = saved[-1]
    WEIGHTS, CONFIG = RUN_DIR / "best.weights.h5", RUN_DIR / "config.json"
    config = json.loads(CONFIG.read_text(encoding="utf-8"))
    vocabulary = config["vocabulary"]
print("모델 폴더:", RUN_DIR)
if MODE == "train":
    import pandas as pd
    from IPython.display import display, Image
    report = json.loads((PREPARED_DIR / "analysis.json").read_text(encoding="utf-8"))
    display(pd.DataFrame(report["data_sources"]).drop(columns="sha256", errors="ignore"))
    display(pd.DataFrame({field: report[field] for field in ["Q", "A"]}).T)
    print("동일 질문에 여러 답변이 있는 질문 수:", report["multiple_answer_questions"])
    display(Image(filename=str(PREPARED_DIR / "length_distribution.png")))
    for field in ["Q", "A"]:
        print(field, "빈도 상위 단어 (표면형 집계, 형태소 분석 아님)")
        display(pd.DataFrame(report["top_words"][field], columns=["단어", "빈도"]))

## 2. 모델 구성
Encoder의 상태를 Decoder 초기 상태로 전달합니다. 학습과 추론이 동일한 레이어를 공유합니다.

In [ ]:
def build_models(config):
    size = len(config["vocabulary"]) + 4
    units, dim = config["units"], config["embedding_dim"]
    encoder_tokens = tf.keras.Input(shape=(None,), dtype="int32", name="encoder_tokens")
    enc_embedding = tf.keras.layers.Embedding(size, dim, mask_zero=True)
    enc_gru = tf.keras.layers.GRU(units, return_state=True)
    _, state = enc_gru(enc_embedding(encoder_tokens))
    decoder_tokens = tf.keras.Input(shape=(None,), dtype="int32", name="decoder_tokens")
    dec_embedding = tf.keras.layers.Embedding(size, dim, mask_zero=True)
    dec_gru = tf.keras.layers.GRU(units, return_sequences=True, return_state=True)
    projection = tf.keras.layers.Dense(size)
    sequence, _ = dec_gru(dec_embedding(decoder_tokens), initial_state=state)
    training = tf.keras.Model([encoder_tokens, decoder_tokens], projection(sequence))
    encoder = tf.keras.Model(encoder_tokens, state)
    previous_state = tf.keras.Input(shape=(units,), name="previous_state")
    sequence, next_state = dec_gru(dec_embedding(decoder_tokens), initial_state=previous_state)
    decoder = tf.keras.Model([decoder_tokens, previous_state], [projection(sequence), next_state])
    return training, encoder, decoder

def masked_loss(actual, predicted):
    loss = tf.keras.losses.sparse_categorical_crossentropy(actual, predicted, from_logits=True)
    mask = tf.cast(tf.not_equal(actual, PAD), loss.dtype)
    return tf.math.divide_no_nan(tf.reduce_sum(loss * mask), tf.reduce_sum(mask))

class MaskedTokenAccuracy(tf.keras.metrics.SparseCategoricalAccuracy):
    """전체 비-PAD 정답 토큰에 대한 정확도 (EOS 포함)."""
    def __init__(self, name="accuracy", **kwargs):
        super().__init__(name=name, **kwargs)

    def update_state(self, y_true, y_pred, sample_weight=None):
        weights = tf.cast(tf.not_equal(y_true, PAD), self.dtype)
        if sample_weight is not None:
            sample_weight = tf.cast(sample_weight, self.dtype)
            if sample_weight.shape.rank == 1:
                sample_weight = tf.expand_dims(sample_weight, -1)
            weights *= sample_weight
        return super().update_state(y_true, y_pred, sample_weight=weights)


model, encoder, decoder = build_models(config)
model.compile(optimizer=tf.keras.optimizers.Adam(clipnorm=1.0), loss=masked_loss, metrics=[MaskedTokenAccuracy()])
model.summary()

## 3. 학습 및 저장
검증 손실이 가장 낮은 가중치를 저장합니다. chat 모드에서는 이 단계를 건너뜁니다.
`loss`, `accuracy`, `val_loss`, `val_accuracy`를 함께 표시하고 history.json에 저장합니다. 정확도는 PAD를 제외하고 EOS를 포함한 문자 토큰 기준이며, 답변의 사실성 점수가 아닙니다. 최적 모델 선택 기준은 val_loss입니다.


In [ ]:
if MODE == "train":
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(str(WEIGHTS), monitor="val_loss", save_best_only=True, save_weights_only=True),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        tf.keras.callbacks.TerminateOnNaN(),
    ]
    history = model.fit(
        [x[:split], decoder_inputs[:split]], y[:split],
        validation_data=([x[split:], decoder_inputs[split:]], y[split:]),
        batch_size=BATCH_SIZE, epochs=EPOCHS, callbacks=callbacks,
    )
    (RUN_DIR / "history.json").write_text(json.dumps(history.history, indent=2), encoding="utf-8")
    print("저장:", WEIGHTS)
if MODE == "train":
    import matplotlib.pyplot as plt
    from IPython.display import display, Image
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    for ax, metric in zip(axes, ["loss", "accuracy"]):
        epochs = range(1, len(history.history[metric]) + 1)
        ax.plot(epochs, history.history[metric], label="train")
        ax.plot(epochs, history.history["val_" + metric], label="validation")
        ax.set(xlabel="Epoch", ylabel=metric, title=metric)
        ax.legend()
    fig.savefig(RUN_DIR / "training_curves.png", dpi=140)
    plt.close(fig)
    display(Image(filename=str(RUN_DIR / "training_curves.png")))

## 4. 디스크에서 모델을 새로 불러오기
메모리의 학습 모델 대신 새 모델에 저장 가중치를 로드합니다. config.json과 best.weights.h5를 함께 보관하세요.

In [ ]:
config = json.loads(CONFIG.read_text(encoding="utf-8"))
vocabulary = config["vocabulary"]
model, encoder, decoder = build_models(config)
model.load_weights(str(WEIGHTS))
reverse_vocabulary = {index: char for char, index in vocabulary.items()}

def chat(question):
    if not question.strip():
        return "질문을 입력해주세요."
    encoded = np.asarray([encode(question, vocabulary, config["max_length"])], dtype="int32")
    state = encoder(encoded, training=False)
    token = np.asarray([[SOS]], dtype="int32")
    answer = []
    for _ in range(config["max_length"]):
        logits, state = decoder([tf.convert_to_tensor(token), state], training=False)
        scores = logits.numpy()[0, -1].copy()
        scores[[PAD, SOS, UNK]] = -np.inf
        index = int(np.argmax(scores))
        if index == EOS:
            break
        answer.append(reverse_vocabulary[index])
        token = np.asarray([[index]], dtype="int32")
    return "".join(answer).strip() or "(모델이 빈 답변을 생성했습니다.)"

print("저장 모델 로드 완료:", WEIGHTS)

## 5. 질문 테스트
아래 질문을 바꾸어 실행하세요. 학습 손실 감소만으로 답변 품질이 보장되지는 않습니다.

In [ ]:
for question in ["안녕", "ESG란 무엇인가요?", "온실가스 배출량은 어떻게 관리하나요?"]:
    print("나:", question)
    print("챗봇:", chat(question))

### 선택: 연속 대화
아래 값을 True로 바꾸면 입력 창이 열립니다. /quit으로 종료합니다. 이전 대화는 기억하지 않습니다.

In [ ]:
INTERACTIVE = False
if INTERACTIVE:
    while True:
        question = input("나: ")
        if question.strip().lower() in {"/quit", "/exit"}:
            break
        print("챗봇:", chat(question))